# Notebook 02 — Construção e Seleção do Target

Este notebook transforma eventos de pagamento em parcelas econômicas, calcula atraso de forma observável e compara definições de inadimplência. A escolha do target considera interpretação de negócio, cobertura, maturidade e capacidade de discriminar risco futuro.

**Princípios**

- não atribuir performance a contratos sem parcelas observáveis;
- consolidar versões e eventos antes de somar valores;
- respeitar censura temporal;
- justificar a definição escolhida com evidência, não apenas convenção.

## 1. Configuração, leitura e parâmetros

Os limites de materialidade evitam classificar como inadimplente uma parcela com diferença residual operacionalmente irrelevante. Eles são explícitos para facilitar auditoria e ajuste.

A data de corte do histórico é parametrizada para representar o encerramento da janela disponível na extração. Contratos sem maturidade suficiente permanecem censurados.

In [1]:
from pathlib import Path
import os
import sys

from pyspark.sql import Row, SparkSession, Window
from pyspark.sql import functions as F

# Garante que driver e workers do Spark utilizem o Python do kernel ativo.
PYTHON_EXECUTAVEL = str(Path(sys.executable).resolve())

os.environ["PYSPARK_PYTHON"] = PYTHON_EXECUTAVEL
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_EXECUTAVEL
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

try:
    spark.stop()
except (NameError, AttributeError):
    pass

spark = (
    SparkSession.builder
    .master(os.getenv("SPARK_MASTER", "local[2]"))
    .appName("credit-risk-case")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", PYTHON_EXECUTAVEL)
    .config("spark.pyspark.driver.python", PYTHON_EXECUTAVEL)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Ajuste manual opcional. Na ausência de valor, são utilizadas variáveis de ambiente,
# um Volume padrão do Databricks, quando disponível, ou pastas locais relativas.
CAMINHO_DADOS_MANUAL = None
CAMINHO_SAIDA_MANUAL = None


def resolver_caminho(caminho_manual, variavel_ambiente, caminho_databricks, caminho_local):
    if caminho_manual:
        return str(caminho_manual)
    if os.getenv(variavel_ambiente):
        return os.environ[variavel_ambiente]
    if caminho_databricks and Path(caminho_databricks).exists():
        return caminho_databricks
    return caminho_local


def juntar_caminho(diretorio, arquivo):
    return f"{str(diretorio).rstrip('/')}/{arquivo}"


DATA_PATH = resolver_caminho(
    CAMINHO_DADOS_MANUAL,
    "CREDIT_RISK_DATA_PATH",
    "/Volumes/workspace/default/credit_risk_data",
    "data"
)

OUTPUT_PATH = resolver_caminho(
    CAMINHO_SAIDA_MANUAL,
    "CREDIT_RISK_OUTPUT_PATH",
    "/Volumes/workspace/default/case_tecnico_ds",
    "outputs"
)

if "://" not in OUTPUT_PATH:
    Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)


def visualizar(df, n=20, truncate=False):
    if n <= 0:
        raise ValueError("O parâmetro 'n' deve ser maior que zero.")
    df.show(n=n, truncate=truncate)


print(f"Python: {sys.version.split()[0]} | Spark: {spark.version}")
print(f"Dados: {DATA_PATH}")
print(f"Saídas: {OUTPUT_PATH}")

Python: 3.11.7 | Spark: 4.1.2
Dados: data
Saídas: outputs


In [2]:
historico_emprestimos = spark.read.parquet(
    juntar_caminho(DATA_PATH, "historico_emprestimos.parquet")
)
historico_parcelas = spark.read.parquet(juntar_caminho(DATA_PATH, "historico_parcelas.parquet"))

TOLERANCIA_PAGAMENTO = 0.01
LIMITE_SALDO_RESIDUAL_ABSOLUTO = 5.00
LIMITE_SALDO_RESIDUAL_PERCENTUAL = 0.001
DATA_CORTE_HISTORICO = "2025-02-22"

CHAVES_PARCELA = ["id_contrato", "id_cliente", "numero_parcela", "data_prevista_pagamento"]
CHAVES_PARCELA_VERSAO = [*CHAVES_PARCELA, "versao_parcela"]

## 2. Consolidação da parcela econômica

Uma parcela pode possuir múltiplas versões e mais de um evento de pagamento. A consolidação separa:

1. valor previsto, agregado no nível da parcela econômica;
2. eventos de pagamento, deduplicados entre versões;
3. pagamento acumulado, usado para identificar a primeira data em que a obrigação foi liquidada;
4. saldo residual, com regra explícita de materialidade.

In [3]:
eventos_pagamento_resumo = historico_parcelas.select(
    F.count("*").alias("qtd_registros"),
    F.sum(F.when(F.col("data_real_pagamento").isNull(), 1).otherwise(0)).alias(
        "qtd_data_pagamento_nula"
    ),
)

visualizar(eventos_pagamento_resumo)

eventos_por_parcela = historico_parcelas.groupBy(*CHAVES_PARCELA_VERSAO).agg(
    F.count("*").alias("qtd_eventos_pagamento")
)
distribuicao_eventos = (
    eventos_por_parcela.groupBy("qtd_eventos_pagamento")
    .agg(F.count("*").alias("qtd_parcelas"))
    .orderBy("qtd_eventos_pagamento")
)

visualizar(distribuicao_eventos)

distribuicao_versoes = (
    historico_parcelas.groupBy("versao_parcela")
    .agg(F.count("*").alias("quantidade"))
    .orderBy("versao_parcela")
)

visualizar(distribuicao_versoes)

parcelas_com_multiplas_versoes = (
    historico_parcelas.groupBy("id_contrato", "numero_parcela")
    .agg(F.countDistinct("versao_parcela").alias("qtd_versoes"))
    .filter(F.col("qtd_versoes") > 1)
)

visualizar(parcelas_com_multiplas_versoes.orderBy(F.desc("qtd_versoes")))

parcelas_por_versao = historico_parcelas.groupBy(*CHAVES_PARCELA_VERSAO).agg(
    F.max("valor_previsto_parcela").alias("valor_previsto_versao"),
    F.sum("valor_pago_parcela").alias("valor_pago_versao"),
    F.max("data_real_pagamento").alias("ultima_data_pagamento_versao"),
    F.count("*").alias("qtd_registros_versao"),
)

validacao_versoes = (
    parcelas_por_versao.groupBy(*CHAVES_PARCELA)
    .agg(
        F.countDistinct("versao_parcela").alias("qtd_versoes"),
        F.sum("valor_previsto_versao").alias("valor_previsto_total"),
        F.max("valor_pago_versao").alias("valor_pago_referencia"),
    )
    .filter(F.col("qtd_versoes") > 1)
    .withColumn(
        "diferenca_pagamento", F.col("valor_pago_referencia") - F.col("valor_previsto_total")
    )
)

resumo_validacao_versoes = validacao_versoes.select(
    F.count("*").alias("grupos_com_multiplas_versoes"),
    F.sum(F.when(F.col("valor_pago_referencia").isNull(), 1).otherwise(0)).alias(
        "grupos_sem_pagamento"
    ),
    F.sum(
        F.when(F.abs(F.col("diferenca_pagamento")) <= F.lit(TOLERANCIA_PAGAMENTO), 1).otherwise(0)
    ).alias("grupos_aderentes"),
)

visualizar(resumo_validacao_versoes)

parcelas_com_multiplos_vencimentos = (
    historico_parcelas.groupBy("id_contrato", "id_cliente", "numero_parcela")
    .agg(
        F.countDistinct("data_prevista_pagamento").alias("qtd_vencimentos"),
        F.countDistinct("versao_parcela").alias("qtd_versoes"),
    )
    .filter(F.col("qtd_vencimentos") > 1)
)

resumo_multiplos_vencimentos = parcelas_com_multiplos_vencimentos.select(
    F.count("*").alias("qtd_parcelas_com_multiplos_vencimentos")
)

visualizar(resumo_multiplos_vencimentos)

+-------------+-----------------------+
|qtd_registros|qtd_data_pagamento_nula|
+-------------+-----------------------+
|1390978      |339                    |
+-------------+-----------------------+

+---------------------+------------+
|qtd_eventos_pagamento|qtd_parcelas|
+---------------------+------------+
|1                    |1252353     |
|2                    |67313       |
|3                    |1211        |
|4                    |63          |
|5                    |9           |
|6                    |9           |
|7                    |1           |
|8                    |1           |
+---------------------+------------+

+--------------+----------+
|versao_parcela|quantidade|
+--------------+----------+
|0.0           |340028    |
|1.0           |936423    |
|2.0           |67502     |
|3.0           |26506     |
|4.0           |6430      |
|5.0           |5399      |
|6.0           |1832      |
|7.0           |1948      |
|8.0           |888       |
|9.0           |91

In [4]:
parcelas_eventos = (
    historico_parcelas.withColumn(
        "data_prevista_pagamento", F.to_date(F.col("data_prevista_pagamento"), "yyyy-MM-dd")
    )
    .withColumn("data_real_pagamento", F.to_date(F.col("data_real_pagamento"), "yyyy-MM-dd"))
    .withColumn("valor_previsto_parcela", F.col("valor_previsto_parcela").cast("double"))
    .withColumn("valor_pago_parcela", F.col("valor_pago_parcela").cast("double"))
)

valores_previstos_por_versao = parcelas_eventos.groupBy(*CHAVES_PARCELA_VERSAO).agg(
    F.max("valor_previsto_parcela").alias("valor_previsto_versao"),
    F.count("*").alias("qtd_registros_versao"),
)

parcelas_previstas = valores_previstos_por_versao.groupBy(*CHAVES_PARCELA).agg(
    F.sum("valor_previsto_versao").alias("valor_previsto_parcela"),
    F.countDistinct("versao_parcela").alias("qtd_versoes"),
    F.sum("qtd_registros_versao").alias("qtd_registros_origem"),
)

eventos_pagamento_deduplicados = (
    parcelas_eventos.filter(
        F.col("data_real_pagamento").isNotNull() & F.col("valor_pago_parcela").isNotNull()
    )
    .groupBy(*CHAVES_PARCELA, "data_real_pagamento", "valor_pago_parcela")
    .agg(
        F.count("*").alias("qtd_registros_evento"),
        F.countDistinct("versao_parcela").alias("qtd_versoes_evento"),
    )
    .withColumnRenamed("valor_pago_parcela", "valor_pago_evento")
)

pagamentos_por_data = eventos_pagamento_deduplicados.groupBy(
    *CHAVES_PARCELA, "data_real_pagamento"
).agg(
    F.sum("valor_pago_evento").alias("valor_pago_data"),
    F.count("*").alias("qtd_eventos_pagamento_data"),
    F.sum("qtd_registros_evento").alias("qtd_registros_pagamento_data"),
)

pagamentos_com_previsto = pagamentos_por_data.join(
    parcelas_previstas.select(*CHAVES_PARCELA, "valor_previsto_parcela"),
    on=CHAVES_PARCELA,
    how="left",
)

In [5]:
janela_pagamentos = (
    Window.partitionBy(*CHAVES_PARCELA)
    .orderBy("data_real_pagamento")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

saldo_residual = F.col("valor_previsto_parcela") - F.col("valor_pago_acumulado")

percentual_saldo_residual = F.when(
    F.col("valor_previsto_parcela") > 0,
    saldo_residual / F.col("valor_previsto_parcela")
)

pagamento_integral = (
    F.col("valor_pago_acumulado") + F.lit(TOLERANCIA_PAGAMENTO)
    >= F.col("valor_previsto_parcela")
)

saldo_residual_aceitavel = (
    (saldo_residual > 0)
    & (saldo_residual <= F.lit(LIMITE_SALDO_RESIDUAL_ABSOLUTO))
    & (percentual_saldo_residual <= F.lit(LIMITE_SALDO_RESIDUAL_PERCENTUAL))
)

condicao_quitacao = (
    (F.col("valor_previsto_parcela") > 0)
    & (pagamento_integral | saldo_residual_aceitavel)
)

pagamentos_acumulados = (
    pagamentos_com_previsto
    .withColumn("valor_pago_acumulado", F.sum("valor_pago_data").over(janela_pagamentos))
    .withColumn("saldo_residual_acumulado", saldo_residual)
    .withColumn("percentual_saldo_residual_acumulado", percentual_saldo_residual)
    .withColumn(
        "data_quitacao_candidata",
        F.when(condicao_quitacao, F.col("data_real_pagamento"))
    )
)

resumo_pagamentos_parcela = (
    pagamentos_acumulados
    .groupBy(*CHAVES_PARCELA)
    .agg(
        F.sum("valor_pago_data").alias("valor_pago_parcela"),
        F.min("data_quitacao_candidata").alias("primeira_data_atingiu_previsto"),
        F.max("data_real_pagamento").alias("data_ultimo_pagamento"),
        F.sum("qtd_eventos_pagamento_data").alias("qtd_eventos_pagamento"),
        F.sum("qtd_registros_pagamento_data").alias("qtd_registros_pagamento")
    )
)



In [6]:
parcelas_consolidadas = (
    parcelas_previstas.join(resumo_pagamentos_parcela, on=CHAVES_PARCELA, how="left")
    .fillna({"valor_pago_parcela": 0.0, "qtd_eventos_pagamento": 0, "qtd_registros_pagamento": 0})
    .withColumn(
        "saldo_residual_bruto", F.col("valor_previsto_parcela") - F.col("valor_pago_parcela")
    )
    .withColumn(
        "percentual_saldo_residual",
        F.when(
            F.col("valor_previsto_parcela") > F.lit(0),
            F.col("saldo_residual_bruto") / F.col("valor_previsto_parcela"),
        ),
    )
    .withColumn(
        "saldo_residual_imaterial",
        F.when(
            (F.col("saldo_residual_bruto") > F.lit(0))
            & (F.col("saldo_residual_bruto") <= F.lit(LIMITE_SALDO_RESIDUAL_ABSOLUTO))
            & (F.col("percentual_saldo_residual") <= F.lit(LIMITE_SALDO_RESIDUAL_PERCENTUAL)),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "data_quitacao",
        F.when(
            (F.col("valor_previsto_parcela") > F.lit(0))
            & (
                (
                    F.col("valor_pago_parcela") + F.lit(TOLERANCIA_PAGAMENTO)
                    >= F.col("valor_previsto_parcela")
                )
                | (F.col("saldo_residual_imaterial") == 1)
            ),
            F.col("primeira_data_atingiu_previsto"),
        ),
    )
    .withColumn(
        "saldo_aberto",
        F.when(
            (
                F.col("valor_pago_parcela") + F.lit(TOLERANCIA_PAGAMENTO)
                >= F.col("valor_previsto_parcela")
            )
            | (F.col("saldo_residual_imaterial") == 1),
            F.lit(0.0),
        ).otherwise(F.greatest(F.col("saldo_residual_bruto"), F.lit(0.0))),
    )
    .withColumn(
        "valor_pago_excedente",
        F.greatest(F.col("valor_pago_parcela") - F.col("valor_previsto_parcela"), F.lit(0.0)),
    )
    .withColumn(
        "status_pagamento",
        F.when(
            F.col("valor_previsto_parcela").isNull()
            | (F.col("valor_previsto_parcela") <= F.lit(0)),
            "Valor previsto inválido",
        )
        .when(
            (
                F.col("valor_pago_parcela") + F.lit(TOLERANCIA_PAGAMENTO)
                >= F.col("valor_previsto_parcela")
            )
            | (F.col("saldo_residual_imaterial") == 1),
            "Quitada",
        )
        .when(F.col("valor_pago_parcela") > F.lit(TOLERANCIA_PAGAMENTO), "Parcial")
        .otherwise("Sem pagamento"),
    )
    .drop("primeira_data_atingiu_previsto")
)

In [7]:
resumo_consolidacao = parcelas_consolidadas.agg(
    F.count("*").alias("qtd_parcelas_economicas"),
    F.countDistinct(F.struct(*CHAVES_PARCELA)).alias("qtd_chaves_distintas"),
    F.sum((F.col("status_pagamento") == "Quitada").cast("int")).alias("qtd_quitadas"),
    F.sum((F.col("status_pagamento") == "Parcial").cast("int")).alias("qtd_parciais"),
    F.sum((F.col("status_pagamento") == "Sem pagamento").cast("int")).alias("qtd_sem_pagamento"),
    F.sum((F.col("status_pagamento") == "Valor previsto inválido").cast("int")).alias(
        "qtd_valor_previsto_invalido"
    ),
    F.sum(
        ((F.col("status_pagamento") == "Quitada") & F.col("data_quitacao").isNull()).cast("int")
    ).alias("qtd_quitadas_sem_data"),
)
visualizar(resumo_consolidacao)

+-----------------------+--------------------+------------+------------+-----------------+---------------------------+---------------------+
|qtd_parcelas_economicas|qtd_chaves_distintas|qtd_quitadas|qtd_parciais|qtd_sem_pagamento|qtd_valor_previsto_invalido|qtd_quitadas_sem_data|
+-----------------------+--------------------+------------+------------+-----------------+---------------------------+---------------------+
|1310965                |1310965             |1310327     |301         |333              |4                          |0                    |
+-----------------------+--------------------+------------+------------+-----------------+---------------------------+---------------------+



**Decisão**

A data real de um pagamento isolado não representa necessariamente a quitação. O atraso de uma parcela parcialmente paga é contado até a data de corte, enquanto parcelas liquidadas usam a primeira data em que o pagamento acumulado atingiu o valor previsto, observada a materialidade definida.

## 3. Dias de atraso e observabilidade

Uma parcela é observável quando possui valor previsto válido e vencimento até a data de corte. Para parcelas quitadas, o atraso termina na quitação; para parcelas parciais ou sem pagamento, termina no corte do histórico.

In [8]:
parcelas_com_atraso = (
    parcelas_consolidadas.withColumn("data_corte_historico", F.to_date(F.lit(DATA_CORTE_HISTORICO)))
    .withColumn(
        "parcela_observavel",
        F.when(
            (F.col("status_pagamento") != "Valor previsto inválido")
            & (F.col("data_prevista_pagamento") <= F.col("data_corte_historico")),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "dias_atraso",
        F.when(F.col("parcela_observavel") == 0, F.lit(None).cast("integer"))
        .when(
            F.col("status_pagamento") == "Quitada",
            F.greatest(
                F.datediff(F.col("data_quitacao"), F.col("data_prevista_pagamento")), F.lit(0)
            ),
        )
        .when(
            F.col("status_pagamento").isin("Parcial", "Sem pagamento"),
            F.greatest(
                F.datediff(F.col("data_corte_historico"), F.col("data_prevista_pagamento")),
                F.lit(0),
            ),
        ),
    )
)

In [9]:
resumo_atraso = parcelas_com_atraso.agg(
    F.count("*").alias("qtd_parcelas"),
    F.sum("parcela_observavel").alias("qtd_observaveis"),
    F.sum((F.col("dias_atraso") > 0).cast("int")).alias("qtd_com_atraso"),
    F.percentile_approx("dias_atraso", 0.50).alias("p50_dias_atraso"),
    F.percentile_approx("dias_atraso", 0.95).alias("p95_dias_atraso"),
    F.percentile_approx("dias_atraso", 0.99).alias("p99_dias_atraso"),
    F.max("dias_atraso").alias("max_dias_atraso"),
)
visualizar(resumo_atraso)

+------------+---------------+--------------+---------------+---------------+---------------+---------------+
|qtd_parcelas|qtd_observaveis|qtd_com_atraso|p50_dias_atraso|p95_dias_atraso|p99_dias_atraso|max_dias_atraso|
+------------+---------------+--------------+---------------+---------------+---------------+---------------+
|1310965     |1310961        |114570        |0              |3              |17             |2573           |
+------------+---------------+--------------+---------------+---------------+---------------+---------------+



## 4. População elegível

A população supervisionada é formada por contratos com ao menos uma parcela econômica válida. Contratos sem performance observável permanecem fora do target; não é criado rótulo artificial para aprovações sem parcelas ou recusas.

In [10]:
resumo_parcelas_contrato = (
    parcelas_com_atraso.filter(F.col("status_pagamento") != "Valor previsto inválido")
    .groupBy("id_contrato", "id_cliente")
    .agg(
        F.min("data_prevista_pagamento").alias("data_primeiro_vencimento_target"),
        F.max("data_prevista_pagamento").alias("data_ultimo_vencimento_target"),
        F.count("*").alias("qtd_parcelas_validas"),
    )
)

validacao_resumo_parcelas_contrato = resumo_parcelas_contrato.select(
    F.count("*").alias("qtd_linhas"),
    F.countDistinct("id_contrato").alias("qtd_contratos_distintos"),
    F.countDistinct("id_cliente").alias("qtd_clientes_distintos"),
    F.sum(F.when(F.col("data_primeiro_vencimento_target").isNull(), 1).otherwise(0)).alias(
        "contratos_sem_primeiro_vencimento"
    ),
    F.min("data_primeiro_vencimento_target").alias("menor_primeiro_vencimento"),
    F.max("data_primeiro_vencimento_target").alias("maior_primeiro_vencimento"),
)

visualizar(validacao_resumo_parcelas_contrato)

populacao_base_targets = resumo_parcelas_contrato.join(
    historico_emprestimos.select(
        "id_contrato", "id_cliente", "status_contrato", "data_decisao", "data_liberacao"
    ),
    on=["id_contrato", "id_cliente"],
    how="inner",
).filter(F.col("status_contrato") == "Approved")

distribuicao_status_populacao = (
    populacao_base_targets.groupBy("status_contrato")
    .agg(
        F.count("*").alias("quantidade"),
        F.countDistinct("id_contrato").alias("qtd_contratos_distintos"),
        F.countDistinct("id_cliente").alias("qtd_clientes_distintos"),
    )
    .orderBy(F.desc("quantidade"))
)

visualizar(distribuicao_status_populacao)

+----------+-----------------------+----------------------+---------------------------------+-------------------------+-------------------------+
|qtd_linhas|qtd_contratos_distintos|qtd_clientes_distintos|contratos_sem_primeiro_vencimento|menor_primeiro_vencimento|maior_primeiro_vencimento|
+----------+-----------------------+----------------------+---------------------------------+-------------------------+-------------------------+
|107419    |107419                 |37739                 |0                                |2017-02-28               |2025-02-20               |
+----------+-----------------------+----------------------+---------------------------------+-------------------------+-------------------------+

+---------------+----------+-----------------------+----------------------+
|status_contrato|quantidade|qtd_contratos_distintos|qtd_clientes_distintos|
+---------------+----------+-----------------------+----------------------+
|Approved       |107419    |107419       

## 5. Targets candidatos

### 5.1 FPD — First Payment Default

O FPD é calculado sobre a primeira obrigação econômica. Quando duas parcelas compartilham o primeiro vencimento, ambas são consolidadas no nível do contrato para evitar escolha arbitrária.

In [11]:
parcelas_primeiro_vencimento = (
    parcelas_com_atraso.alias("p")
    .join(
        populacao_base_targets.select(
            "id_contrato", "id_cliente", "data_primeiro_vencimento_target"
        ).alias("c"),
        on=["id_contrato", "id_cliente"],
        how="inner",
    )
    .filter(F.col("p.data_prevista_pagamento") == F.col("c.data_primeiro_vencimento_target"))
    .select(
        "id_contrato",
        "id_cliente",
        F.col("c.data_primeiro_vencimento_target"),
        F.col("p.numero_parcela"),
        F.col("p.data_prevista_pagamento"),
        F.col("p.data_quitacao"),
        F.col("p.status_pagamento"),
        F.col("p.dias_atraso"),
        F.col("p.valor_previsto_parcela"),
        F.col("p.valor_pago_parcela"),
        F.col("p.saldo_aberto"),
    )
)

resumo_base_fpd = (
    parcelas_primeiro_vencimento.groupBy("id_contrato", "id_cliente")
    .agg(
        F.count("*").alias("qtd_parcelas_primeiro_vencimento"),
        F.max("dias_atraso").alias("maior_atraso_primeiro_vencimento"),
    )
    .select(
        F.count("*").alias("qtd_contratos"),
        F.sum(F.when(F.col("qtd_parcelas_primeiro_vencimento") > 1, 1).otherwise(0)).alias(
            "contratos_com_multiplas_parcelas_primeiro_vencimento"
        ),
        F.max("qtd_parcelas_primeiro_vencimento").alias("max_parcelas_primeiro_vencimento"),
        F.sum(F.when(F.col("maior_atraso_primeiro_vencimento").isNull(), 1).otherwise(0)).alias(
            "contratos_sem_atraso_calculado"
        ),
    )
)

visualizar(resumo_base_fpd)

base_fpd_contrato = (
    parcelas_primeiro_vencimento.groupBy(
        "id_contrato", "id_cliente", "data_primeiro_vencimento_target"
    )
    .agg(
        F.count("*").alias("qtd_parcelas_primeiro_vencimento"),
        F.max("dias_atraso").alias("maior_atraso_primeiro_vencimento"),
        F.sum("valor_previsto_parcela").alias("valor_previsto_primeiro_vencimento"),
        F.sum("valor_pago_parcela").alias("valor_pago_primeiro_vencimento"),
        F.sum("saldo_aberto").alias("saldo_aberto_primeiro_vencimento"),
    )
    .withColumn(
        "dias_observacao_primeiro_vencimento",
        F.datediff(
            F.to_date(F.lit(DATA_CORTE_HISTORICO)), F.col("data_primeiro_vencimento_target")
        ),
    )
)

total_contratos_base_fpd = base_fpd_contrato.count()

distribuicao_atraso_fpd = (
    base_fpd_contrato.withColumn(
        "ordem_faixa",
        F.when(F.col("maior_atraso_primeiro_vencimento") == 0, 1)
        .when(F.col("maior_atraso_primeiro_vencimento") <= 5, 2)
        .when(F.col("maior_atraso_primeiro_vencimento") <= 15, 3)
        .when(F.col("maior_atraso_primeiro_vencimento") <= 30, 4)
        .when(F.col("maior_atraso_primeiro_vencimento") <= 60, 5)
        .when(F.col("maior_atraso_primeiro_vencimento") <= 90, 6)
        .otherwise(7),
    )
    .withColumn(
        "faixa_atraso_fpd",
        F.when(F.col("maior_atraso_primeiro_vencimento") == 0, "Em dia")
        .when(F.col("maior_atraso_primeiro_vencimento") <= 5, "1 a 5 dias")
        .when(F.col("maior_atraso_primeiro_vencimento") <= 15, "6 a 15 dias")
        .when(F.col("maior_atraso_primeiro_vencimento") <= 30, "16 a 30 dias")
        .when(F.col("maior_atraso_primeiro_vencimento") <= 60, "31 a 60 dias")
        .when(F.col("maior_atraso_primeiro_vencimento") <= 90, "61 a 90 dias")
        .otherwise("Acima de 90 dias"),
    )
    .groupBy("ordem_faixa", "faixa_atraso_fpd")
    .agg(F.count("*").alias("quantidade"))
    .withColumn(
        "percentual", F.round(F.col("quantidade") / F.lit(total_contratos_base_fpd) * 100, 4)
    )
    .orderBy("ordem_faixa")
    .drop("ordem_faixa")
)

visualizar(distribuicao_atraso_fpd)

base_fpd = base_fpd_contrato.withColumn(
    "target_fpd_qualquer_atraso",
    F.when(F.col("maior_atraso_primeiro_vencimento").isNull(), F.lit(None).cast("integer"))
    .when(F.col("maior_atraso_primeiro_vencimento") > 0, F.lit(1))
    .otherwise(F.lit(0)),
).withColumn(
    "target_fpd_tolerancia_5d",
    F.when(F.col("maior_atraso_primeiro_vencimento") > 5, F.lit(1))
    .when(F.col("saldo_aberto_primeiro_vencimento") == 0, F.lit(0))
    .otherwise(F.lit(None).cast("integer")),
)

targets_fpd_formato_longo = base_fpd.selectExpr(
    "id_contrato",
    "id_cliente",
    """
        stack(
            2,
            'FPD_QUALQUER_ATRASO',
            target_fpd_qualquer_atraso,
            'FPD_TOLERANCIA_5D',
            target_fpd_tolerancia_5d
        ) AS (
            definicao_target,
            valor_target
        )
        """,
)

totais_targets_fpd = targets_fpd_formato_longo.groupBy("definicao_target").agg(
    F.count("*").alias("total_contratos"), F.count("valor_target").alias("total_classificados")
)

distribuicao_targets_fpd = (
    targets_fpd_formato_longo.groupBy("definicao_target", "valor_target")
    .agg(F.count("*").alias("quantidade"))
    .join(totais_targets_fpd, on="definicao_target", how="left")
    .withColumn(
        "percentual_total", F.round(F.col("quantidade") / F.col("total_contratos") * 100, 4)
    )
    .withColumn(
        "percentual_classificados",
        F.when(
            F.col("valor_target").isNotNull(),
            F.round(F.col("quantidade") / F.col("total_classificados") * 100, 4),
        ),
    )
    .orderBy("definicao_target", "valor_target")
)

visualizar(distribuicao_targets_fpd)

+-------------+----------------------------------------------------+--------------------------------+------------------------------+
|qtd_contratos|contratos_com_multiplas_parcelas_primeiro_vencimento|max_parcelas_primeiro_vencimento|contratos_sem_atraso_calculado|
+-------------+----------------------------------------------------+--------------------------------+------------------------------+
|107419       |221                                                 |2                               |0                             |
+-------------+----------------------------------------------------+--------------------------------+------------------------------+

+----------------+----------+----------+
|faixa_atraso_fpd|quantidade|percentual|
+----------------+----------+----------+
|Em dia          |103349    |96.2111   |
|1 a 5 dias      |2529      |2.3543    |
|6 a 15 dias     |1077      |1.0026    |
|16 a 30 dias    |421       |0.3919    |
|31 a 60 dias    |36        |0.0335    |
|61 a 

### 5.2 EVER30MOB03

O contrato é mau quando alguma parcela ultrapassa 30 dias de atraso dentro dos três primeiros meses. Contratos sem janela completa e sem evento observado permanecem censurados.

In [12]:
elegibilidade_ever30mob03 = (
    populacao_base_targets.withColumn(
        "data_inicio_mob03", F.to_date(F.col("data_decisao"), "yyyy-MM-dd")
    )
    .withColumn("data_fim_mob03", F.add_months(F.col("data_inicio_mob03"), 3))
    .withColumn(
        "contrato_maduro_ever30mob03",
        F.when(F.col("data_fim_mob03") <= F.to_date(F.lit(DATA_CORTE_HISTORICO)), 1).otherwise(0),
    )
)

parcelas_elegiveis_ever30mob03 = (
    parcelas_com_atraso.alias("p")
    .join(
        elegibilidade_ever30mob03.select(
            "id_contrato",
            "id_cliente",
            "data_inicio_mob03",
            "data_fim_mob03",
            "contrato_maduro_ever30mob03",
        ).alias("c"),
        on=["id_contrato", "id_cliente"],
        how="inner",
    )
    .filter(F.col("p.status_pagamento") != "Valor previsto inválido")
    .withColumn("data_atinge_mais_30_dias", F.date_add(F.col("p.data_prevista_pagamento"), 31))
    .filter(
        (F.col("p.data_prevista_pagamento") >= F.col("data_inicio_mob03"))
        & (F.col("data_atinge_mais_30_dias") <= F.col("data_fim_mob03"))
    )
)

contratos_com_parcela_elegivel_mob03 = parcelas_elegiveis_ever30mob03.select(
    "id_contrato", "id_cliente"
).distinct()

validacao_elegibilidade_ever30mob03 = (
    elegibilidade_ever30mob03.alias("c")
    .join(
        contratos_com_parcela_elegivel_mob03.withColumn(
            "possui_parcela_elegivel_mob03", F.lit(1)
        ).alias("p"),
        on=["id_contrato", "id_cliente"],
        how="left",
    )
    .select(
        F.count("*").alias("qtd_contratos"),
        F.sum("contrato_maduro_ever30mob03").alias("contratos_maduros"),
        F.sum(F.when(F.col("possui_parcela_elegivel_mob03") == 1, 1).otherwise(0)).alias(
            "contratos_com_parcela_elegivel"
        ),
        F.sum(
            F.when(
                (F.col("contrato_maduro_ever30mob03") == 1)
                & (F.col("possui_parcela_elegivel_mob03") == 1),
                1,
            ).otherwise(0)
        ).alias("maduros_com_parcela_elegivel"),
        F.sum(
            F.when(
                (F.col("contrato_maduro_ever30mob03") == 1)
                & (F.col("possui_parcela_elegivel_mob03").isNull()),
                1,
            ).otherwise(0)
        ).alias("maduros_sem_parcela_elegivel"),
    )
)

visualizar(validacao_elegibilidade_ever30mob03)

eventos_ever30mob03_contrato = (
    parcelas_elegiveis_ever30mob03.withColumn(
        "flag_evento_observado_ever30mob03",
        F.when(
            (F.col("data_atinge_mais_30_dias") <= F.to_date(F.lit(DATA_CORTE_HISTORICO)))
            & (
                F.col("data_quitacao").isNull()
                | (F.col("data_quitacao") >= F.col("data_atinge_mais_30_dias"))
            ),
            1,
        ).otherwise(0),
    )
    .groupBy("id_contrato", "id_cliente")
    .agg(
        F.max("flag_evento_observado_ever30mob03").alias("flag_evento_ever30mob03"),
        F.min(
            F.when(
                F.col("flag_evento_observado_ever30mob03") == 1, F.col("data_atinge_mais_30_dias")
            )
        ).alias("primeira_data_evento_ever30mob03"),
    )
)

base_ever30mob03 = (
    elegibilidade_ever30mob03.join(
        eventos_ever30mob03_contrato, on=["id_contrato", "id_cliente"], how="left"
    )
    .withColumn("flag_evento_ever30mob03", F.coalesce(F.col("flag_evento_ever30mob03"), F.lit(0)))
    .withColumn(
        "target_ever30mob03",
        F.when(F.col("flag_evento_ever30mob03") == 1, 1)
        .when(F.col("contrato_maduro_ever30mob03") == 1, 0)
        .otherwise(F.lit(None).cast("integer")),
    )
)

total_contratos_ever30mob03 = base_ever30mob03.count()

total_classificados_ever30mob03 = base_ever30mob03.filter(
    F.col("target_ever30mob03").isNotNull()
).count()

distribuicao_target_ever30mob03 = (
    base_ever30mob03.groupBy("target_ever30mob03")
    .agg(F.count("*").alias("quantidade"))
    .withColumn(
        "percentual_total",
        F.round(F.col("quantidade") / F.lit(total_contratos_ever30mob03) * 100, 4),
    )
    .withColumn(
        "percentual_classificados",
        F.when(
            F.col("target_ever30mob03").isNotNull(),
            F.round(F.col("quantidade") / F.lit(total_classificados_ever30mob03) * 100, 4),
        ),
    )
    .orderBy("target_ever30mob03")
)

visualizar(distribuicao_target_ever30mob03)

validacao_maturidade_target_ever30mob03 = (
    base_ever30mob03.groupBy("contrato_maduro_ever30mob03", "target_ever30mob03")
    .agg(F.count("*").alias("quantidade"))
    .orderBy("contrato_maduro_ever30mob03", "target_ever30mob03")
)

visualizar(validacao_maturidade_target_ever30mob03)

+-------------+-----------------+------------------------------+----------------------------+----------------------------+
|qtd_contratos|contratos_maduros|contratos_com_parcela_elegivel|maduros_com_parcela_elegivel|maduros_sem_parcela_elegivel|
+-------------+-----------------+------------------------------+----------------------------+----------------------------+
|107419       |106354           |104390                        |103330                      |3024                        |
+-------------+-----------------+------------------------------+----------------------------+----------------------------+

+------------------+----------+----------------+------------------------+
|target_ever30mob03|quantidade|percentual_total|percentual_classificados|
+------------------+----------+----------------+------------------------+
|NULL              |1064      |0.9905          |NULL                    |
|0                 |106265    |98.9257         |99.9154                 |
|1            

### 5.3 Targets no horizonte MOB06

São avaliadas quatro alternativas adicionais: EVER30, EVER60, OVER30 e OVER60 em seis meses. A maturidade é definida antes da classificação para separar ausência de evento de ausência de observação.

In [13]:
elegibilidade_targets_mob06 = (
    populacao_base_targets.withColumn(
        "data_inicio_mob06", F.to_date(F.col("data_decisao"), "yyyy-MM-dd")
    )
    .withColumn("data_fim_mob06", F.add_months(F.col("data_inicio_mob06"), 6))
    .withColumn(
        "contrato_maduro_mob06",
        F.when(F.col("data_fim_mob06") <= F.to_date(F.lit(DATA_CORTE_HISTORICO)), 1).otherwise(0),
    )
)

validacao_maturidade_targets_mob06 = elegibilidade_targets_mob06.select(
    F.count("*").alias("qtd_contratos"),
    F.sum("contrato_maduro_mob06").alias("contratos_maduros"),
    F.sum(F.when(F.col("contrato_maduro_mob06") == 0, 1).otherwise(0)).alias(
        "contratos_nao_maduros"
    ),
    F.min("data_fim_mob06").alias("menor_data_fim_mob06"),
    F.max("data_fim_mob06").alias("maior_data_fim_mob06"),
)

visualizar(validacao_maturidade_targets_mob06)

parcelas_avaliacao_targets_mob06 = (
    parcelas_com_atraso.alias("p")
    .join(
        elegibilidade_targets_mob06.select(
            "id_contrato",
            "id_cliente",
            "data_inicio_mob06",
            "data_fim_mob06",
            "contrato_maduro_mob06",
        ).alias("c"),
        on=["id_contrato", "id_cliente"],
        how="inner",
    )
    .filter(F.col("p.status_pagamento") != "Valor previsto inválido")
    .filter(
        (F.col("p.data_prevista_pagamento") >= F.col("c.data_inicio_mob06"))
        & (F.col("p.data_prevista_pagamento") <= F.col("c.data_fim_mob06"))
    )
    .select(
        "id_contrato",
        "id_cliente",
        F.col("p.numero_parcela"),
        F.col("p.data_prevista_pagamento"),
        F.col("p.data_quitacao"),
        F.col("p.status_pagamento"),
        F.col("p.valor_previsto_parcela"),
        F.col("p.valor_pago_parcela"),
        F.col("p.saldo_aberto"),
        F.col("c.data_inicio_mob06"),
        F.col("c.data_fim_mob06"),
        F.col("c.contrato_maduro_mob06"),
    )
    .withColumn("data_atinge_mais_30_dias", F.date_add(F.col("data_prevista_pagamento"), 31))
    .withColumn("data_atinge_mais_60_dias", F.date_add(F.col("data_prevista_pagamento"), 61))
    .withColumn(
        "dias_atraso_no_fim_mob06",
        F.greatest(F.datediff(F.col("data_fim_mob06"), F.col("data_prevista_pagamento")), F.lit(0)),
    )
    .withColumn(
        "parcela_aberta_no_fim_mob06",
        F.when(
            F.col("data_quitacao").isNull() | (F.col("data_quitacao") > F.col("data_fim_mob06")), 1
        ).otherwise(0),
    )
)

parcelas_com_flags_targets_mob06 = (
    parcelas_avaliacao_targets_mob06.withColumn(
        "flag_ever30mob06_parcela",
        F.when(
            (F.col("data_atinge_mais_30_dias") <= F.col("data_fim_mob06"))
            & (F.col("data_atinge_mais_30_dias") <= F.to_date(F.lit(DATA_CORTE_HISTORICO)))
            & (
                F.col("data_quitacao").isNull()
                | (F.col("data_quitacao") >= F.col("data_atinge_mais_30_dias"))
            ),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "flag_ever60mob06_parcela",
        F.when(
            (F.col("data_atinge_mais_60_dias") <= F.col("data_fim_mob06"))
            & (F.col("data_atinge_mais_60_dias") <= F.to_date(F.lit(DATA_CORTE_HISTORICO)))
            & (
                F.col("data_quitacao").isNull()
                | (F.col("data_quitacao") >= F.col("data_atinge_mais_60_dias"))
            ),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "flag_over30mob06_parcela",
        F.when(
            (F.col("contrato_maduro_mob06") == 1)
            & (F.col("parcela_aberta_no_fim_mob06") == 1)
            & (F.col("dias_atraso_no_fim_mob06") > 30),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "flag_over60mob06_parcela",
        F.when(
            (F.col("contrato_maduro_mob06") == 1)
            & (F.col("parcela_aberta_no_fim_mob06") == 1)
            & (F.col("dias_atraso_no_fim_mob06") > 60),
            1,
        ).otherwise(0),
    )
)

eventos_targets_mob06_contrato = parcelas_com_flags_targets_mob06.groupBy(
    "id_contrato", "id_cliente"
).agg(
    F.max("flag_ever30mob06_parcela").alias("flag_evento_ever30mob06"),
    F.max("flag_ever60mob06_parcela").alias("flag_evento_ever60mob06"),
    F.max("flag_over30mob06_parcela").alias("flag_evento_over30mob06"),
    F.max("flag_over60mob06_parcela").alias("flag_evento_over60mob06"),
)

base_targets_mob06 = (
    elegibilidade_targets_mob06.join(
        eventos_targets_mob06_contrato, on=["id_contrato", "id_cliente"], how="left"
    )
    .fillna(
        {
            "flag_evento_ever30mob06": 0,
            "flag_evento_ever60mob06": 0,
            "flag_evento_over30mob06": 0,
            "flag_evento_over60mob06": 0,
        }
    )
    .withColumn(
        "target_ever30mob06",
        F.when(F.col("flag_evento_ever30mob06") == 1, 1)
        .when(F.col("contrato_maduro_mob06") == 1, 0)
        .otherwise(F.lit(None).cast("integer")),
    )
    .withColumn(
        "target_ever60mob06",
        F.when(F.col("flag_evento_ever60mob06") == 1, 1)
        .when(F.col("contrato_maduro_mob06") == 1, 0)
        .otherwise(F.lit(None).cast("integer")),
    )
    .withColumn(
        "target_over30mob06",
        F.when(F.col("contrato_maduro_mob06") == 1, F.col("flag_evento_over30mob06")).otherwise(
            F.lit(None).cast("integer")
        ),
    )
    .withColumn(
        "target_over60mob06",
        F.when(F.col("contrato_maduro_mob06") == 1, F.col("flag_evento_over60mob06")).otherwise(
            F.lit(None).cast("integer")
        ),
    )
)

targets_mob06_formato_longo = base_targets_mob06.selectExpr(
    "id_contrato",
    "id_cliente",
    """
        stack(
            4,
            'EVER30MOB06',
            target_ever30mob06,
            'EVER60MOB06',
            target_ever60mob06,
            'OVER30MOB06',
            target_over30mob06,
            'OVER60MOB06',
            target_over60mob06
        ) AS (
            definicao_target,
            valor_target
        )
        """,
)

totais_targets_mob06 = targets_mob06_formato_longo.groupBy("definicao_target").agg(
    F.count("*").alias("total_contratos"), F.count("valor_target").alias("total_classificados")
)

distribuicao_targets_mob06 = (
    targets_mob06_formato_longo.groupBy("definicao_target", "valor_target")
    .agg(F.count("*").alias("quantidade"))
    .join(totais_targets_mob06, on="definicao_target", how="left")
    .withColumn(
        "percentual_total", F.round(F.col("quantidade") / F.col("total_contratos") * 100, 4)
    )
    .withColumn(
        "percentual_classificados",
        F.when(
            F.col("valor_target").isNotNull(),
            F.round(F.col("quantidade") / F.col("total_classificados") * 100, 4),
        ),
    )
    .orderBy("definicao_target", "valor_target")
)

visualizar(distribuicao_targets_mob06)

+-------------+-----------------+---------------------+--------------------+--------------------+
|qtd_contratos|contratos_maduros|contratos_nao_maduros|menor_data_fim_mob06|maior_data_fim_mob06|
+-------------+-----------------+---------------------+--------------------+--------------------+
|107419       |102756           |4663                 |2017-08-04          |2025-08-14          |
+-------------+-----------------+---------------------+--------------------+--------------------+

+----------------+------------+----------+---------------+-------------------+----------------+------------------------+
|definicao_target|valor_target|quantidade|total_contratos|total_classificados|percentual_total|percentual_classificados|
+----------------+------------+----------+---------------+-------------------+----------------+------------------------+
|EVER30MOB06     |NULL        |4657      |107419         |102762             |4.3354          |NULL                    |
|EVER30MOB06     |0      

## 6. Comparação e escolha

As alternativas são comparadas por volume classificado, taxa de maus e censura. Como validação adicional, os atrasos de 1 a 5 dias no primeiro vencimento são relacionados à deterioração posterior no horizonte de seis meses.

In [14]:
targets_ever30mob03_formato_longo = base_ever30mob03.select(
    "id_contrato",
    "id_cliente",
    F.lit("EVER30MOB03").alias("definicao_target"),
    F.col("target_ever30mob03").alias("valor_target"),
)

targets_comparacao_formato_longo = (
    targets_fpd_formato_longo.select(
        "id_contrato", "id_cliente", "definicao_target", "valor_target"
    )
    .unionByName(targets_ever30mob03_formato_longo)
    .unionByName(
        targets_mob06_formato_longo.select(
            "id_contrato", "id_cliente", "definicao_target", "valor_target"
        )
    )
)

resumo_comparacao_targets = (
    targets_comparacao_formato_longo.groupBy("definicao_target")
    .agg(
        F.count("*").alias("total_contratos"),
        F.count("valor_target").alias("total_classificados"),
        F.sum(F.when(F.col("valor_target") == 0, 1).otherwise(0)).alias("qtd_bons"),
        F.sum(F.when(F.col("valor_target") == 1, 1).otherwise(0)).alias("qtd_maus"),
        F.sum(F.when(F.col("valor_target").isNull(), 1).otherwise(0)).alias("qtd_censurados"),
    )
    .withColumn(
        "taxa_maus_classificados", F.round(F.col("qtd_maus") / F.col("total_classificados") * 100, 4)
    )
    .withColumn(
        "percentual_censura", F.round(F.col("qtd_censurados") / F.col("total_contratos") * 100, 4)
    )
    .withColumn(
        "percentual_classificado",
        F.round(F.col("total_classificados") / F.col("total_contratos") * 100, 4),
    )
    .withColumn(
        "ordem_target",
        F.when(F.col("definicao_target") == "FPD_QUALQUER_ATRASO", 1)
        .when(F.col("definicao_target") == "FPD_TOLERANCIA_5D", 2)
        .when(F.col("definicao_target") == "EVER30MOB03", 3)
        .when(F.col("definicao_target") == "EVER30MOB06", 4)
        .when(F.col("definicao_target") == "EVER60MOB06", 5)
        .when(F.col("definicao_target") == "OVER30MOB06", 6)
        .when(F.col("definicao_target") == "OVER60MOB06", 7),
    )
    .orderBy("ordem_target")
    .drop("ordem_target")
)

visualizar(resumo_comparacao_targets)

+-------------------+---------------+-------------------+--------+--------+--------------+-----------------------+------------------+-----------------------+
|definicao_target   |total_contratos|total_classificados|qtd_bons|qtd_maus|qtd_censurados|taxa_maus_classificados|percentual_censura|percentual_classificado|
+-------------------+---------------+-------------------+--------+--------+--------------+-----------------------+------------------+-----------------------+
|FPD_QUALQUER_ATRASO|107419         |107419             |103349  |4070    |0             |3.7889                 |0.0               |100.0                  |
|FPD_TOLERANCIA_5D  |107419         |107419             |105878  |1541    |0             |1.4346                 |0.0               |100.0                  |
|EVER30MOB03        |107419         |106355             |106265  |90      |1064          |0.0846                 |0.9905            |99.0095                |
|EVER30MOB06        |107419         |102762         

In [15]:
analise_atraso_inicial_comportamento_posterior = (
    base_fpd_contrato.select("id_contrato", "id_cliente", "maior_atraso_primeiro_vencimento")
    .join(
        base_targets_mob06.select(
            "id_contrato", "id_cliente", "target_ever30mob06", "target_over30mob06"
        ),
        on=["id_contrato", "id_cliente"],
        how="inner",
    )
    .withColumn(
        "faixa_atraso_primeira_obrigacao",
        F.when(F.col("maior_atraso_primeiro_vencimento") == 0, "Em dia")
        .when(F.col("maior_atraso_primeiro_vencimento").between(1, 5), "Atraso de 1 a 5 dias")
        .otherwise("Atraso superior a 5 dias"),
    )
)

comparacao_atraso_inicial_comportamento_posterior = (
    analise_atraso_inicial_comportamento_posterior.groupBy("faixa_atraso_primeira_obrigacao")
    .agg(
        F.count("*").alias("qtd_contratos"),
        F.count("target_ever30mob06").alias("qtd_classificados_ever30mob06"),
        F.sum(F.when(F.col("target_ever30mob06") == 1, 1).otherwise(0)).alias(
            "qtd_maus_ever30mob06"
        ),
        F.count("target_over30mob06").alias("qtd_classificados_over30mob06"),
        F.sum(F.when(F.col("target_over30mob06") == 1, 1).otherwise(0)).alias(
            "qtd_maus_over30mob06"
        ),
    )
    .withColumn(
        "taxa_maus_ever30mob06",
        F.round(F.col("qtd_maus_ever30mob06") / F.col("qtd_classificados_ever30mob06") * 100, 4),
    )
    .withColumn(
        "taxa_maus_over30mob06",
        F.round(F.col("qtd_maus_over30mob06") / F.col("qtd_classificados_over30mob06") * 100, 4),
    )
    .withColumn(
        "ordem_faixa",
        F.when(F.col("faixa_atraso_primeira_obrigacao") == "Em dia", 1)
        .when(F.col("faixa_atraso_primeira_obrigacao") == "Atraso de 1 a 5 dias", 2)
        .otherwise(3),
    )
    .orderBy("ordem_faixa")
    .drop("ordem_faixa")
)

visualizar(comparacao_atraso_inicial_comportamento_posterior)

+-------------------------------+-------------+-----------------------------+--------------------+-----------------------------+--------------------+---------------------+---------------------+
|faixa_atraso_primeira_obrigacao|qtd_contratos|qtd_classificados_ever30mob06|qtd_maus_ever30mob06|qtd_classificados_over30mob06|qtd_maus_over30mob06|taxa_maus_ever30mob06|taxa_maus_over30mob06|
+-------------------------------+-------------+-----------------------------+--------------------+-----------------------------+--------------------+---------------------+---------------------+
|Em dia                         |103349       |98776                        |274                 |98772                        |80                  |0.2774               |0.081                |
|Atraso de 1 a 5 dias           |2529         |2473                         |42                  |2473                         |14                  |1.6983               |0.5661               |
|Atraso superior a 5 dias     

**Target selecionado: FPD com qualquer atraso na primeira obrigação**

A escolha preserva a interpretação literal do FPD apresentada no case e oferece a maior cobertura entre as alternativas. A análise de comportamento posterior mostrou que atrasos iniciais curtos não são apenas ruído operacional: o grupo de 1 a 5 dias apresentou deterioração futura superior ao grupo adimplente.

O target não pretende afirmar que qualquer atraso curto equivale a perda definitiva. Ele representa risco de falha já na primeira obrigação, um sinal precoce e operacionalmente relevante para concessão. A definição deve ser monitorada em produção e pode ser complementada por targets de maior severidade em estudos futuros.

In [16]:
COLUNA_TARGET_FINAL = "target_fpd_qualquer_atraso"

base_target_final = populacao_base_targets.select("id_contrato", "id_cliente", "data_decisao").join(
    base_fpd.select(
        "id_contrato",
        "id_cliente",
        "data_primeiro_vencimento_target",
        "maior_atraso_primeiro_vencimento",
        "valor_previsto_primeiro_vencimento",
        "valor_pago_primeiro_vencimento",
        "saldo_aberto_primeiro_vencimento",
        F.col(COLUNA_TARGET_FINAL).alias("target"),
    ),
    on=["id_contrato", "id_cliente"],
    how="inner",
)

validacao_target_final = base_target_final.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct(F.struct("id_contrato", "id_cliente")).alias("total_contratos_distintos"),
    F.sum(F.when(F.col("target") == 0, 1).otherwise(0)).alias("qtd_bons"),
    F.sum(F.when(F.col("target") == 1, 1).otherwise(0)).alias("qtd_maus"),
    F.sum(F.when(F.col("target").isNull(), 1).otherwise(0)).alias("qtd_targets_nulas"),
).withColumn("taxa_maus", F.round(F.col("qtd_maus") / F.col("total_registros") * 100, 4))

visualizar(validacao_target_final)

+---------------+-------------------------+--------+--------+-----------------+---------+
|total_registros|total_contratos_distintos|qtd_bons|qtd_maus|qtd_targets_nulas|taxa_maus|
+---------------+-------------------------+--------+--------+-----------------+---------+
|107419         |107419                   |103349  |4070    |0                |3.7889   |
+---------------+-------------------------+--------+--------+-----------------+---------+



## 7. Salvamento e validação das saídas


A saída contém uma linha por contrato elegível, o target final e campos necessários para auditoria. O arquivo intermediário é gerado durante a execução.

In [17]:
def salvar_parquet_local(dataframe_spark, caminho):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)

    dataframe_spark.toPandas().to_parquet(
        caminho,
        index=False,
        engine="pyarrow"
    )


CAMINHO_TARGET_FINAL = juntar_caminho(
    OUTPUT_PATH,
    "base_target_final.parquet"
)

salvar_parquet_local(
    base_target_final,
    CAMINHO_TARGET_FINAL
)

base_target_final_validacao = spark.read.parquet(
    CAMINHO_TARGET_FINAL
)

qtd_registros_original = base_target_final.count()
qtd_registros_persistido = base_target_final_validacao.count()

if qtd_registros_original != qtd_registros_persistido:
    raise ValueError(
        "A base persistida não preservou a quantidade de registros."
    )

if base_target_final.columns != base_target_final_validacao.columns:
    raise ValueError(
        "A base persistida não preservou as colunas esperadas."
    )

print("Registros:", qtd_registros_persistido)
print("Colunas:", len(base_target_final_validacao.columns))
print("Estrutura persistida com sucesso.")

Registros: 107419
Colunas: 9
Estrutura persistida com sucesso.


## 8. Conclusões

- A consolidação financeira evita dupla contagem de versões e pagamentos.
- A regra de atraso distingue parcela quitada, parcial, não paga e não observável.
- A população final contém 107.419 contratos classificados, dos quais 4.070 são maus, com taxa de maus aproximada de 3,79%.
- O FPD com qualquer atraso foi selecionado por cobertura, interpretação e evidência de deterioração posterior.
- Contratos recusados e aprovados sem performance não receberam target artificial.
- O desenvolvimento está sujeito a viés de aprovação, pois somente contratos concedidos e observáveis compõem o target. Inferência de rejeitados é uma evolução possível, não uma suposição aplicada nesta entrega.



## 9. Próximos passos

O Notebook 03 construirá features point-in-time para a população de desenvolvimento e para a base de scoring. O Notebook 04 integrará o target, realizará a validação temporal, comparará modelos, calibrará probabilidades e proporá a política de crédito.